In [1]:
from google.colab import drive
drive.mount('/content/drive')
#'/content/drive/MyDrive/pr/model_task1_state_dict.pt'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pickle
import pandas as pd

In [ ]:
# apply the same transform procedure as the train data
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:

class test_BagDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.bags = []

        for bag in os.listdir(root_dir):
            bag_path = os.path.join(root_dir, bag)
            with open(bag_path, 'rb') as f:
                images = pickle.load(f)
            self.bags.append((bag, images))

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        bag_name, images = self.bags[idx]
        imgs = []

        for img in images:
            image = Image.fromarray(img)
            if self.transform:
                image = self.transform(image)
            imgs.append(image)

        imgs = torch.stack(imgs, dim=0)
        return bag_name, imgs


In [ ]:

class BagNet(nn.Module):
    def __init__(self):
        super(BagNet, self).__init__()
        self.resnet18 = models.resnet18(pretrained=True)
        self.resnet18 = nn.Sequential(*list(self.resnet18.children())[:-1])
        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(512, 128)
        self.fc2 = nn.Linear(128, 1) 

    def forward(self, x):
        batch_size, num_images, C, H, W = x.shape
        x = x.view(batch_size * num_images, C, H, W)
        features = self.resnet18(x)
        features = features.view(batch_size, num_images, -1)
        bag_features = torch.mean(features, dim=1)
        bag_features = self.dropout(bag_features)
        x = self.fc1(bag_features)
        x = torch.relu(x)
        x = self.fc2(x)
        return x.squeeze(1)


In [ ]:
test_root_dir = '/content/drive/MyDrive/pr/test' 
test_dataset = test_BagDataset(root_dir=test_root_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else "cpu")
model = BagNet().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/pr/train_epoch_4.pth')) #lopad the weight to predict
model.eval()

results = []
with torch.no_grad():
    for bag_name, inputs in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        print(outputs)
        preds = (outputs >= 0.5).int()
        results.append((bag_name[0], preds.item()))


submission_df = pd.DataFrame(results, columns=['image_id', 'y_pred'])
submission_df['image_id'] = submission_df['image_id'].str.split('.').str[0] #to delete the .pkl
submission_df.to_csv('submission.csv', index=False)
